<a href="https://colab.research.google.com/github/mehedihasan-cse/Open-Source/blob/main/GDELT_DATA_LArge_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df =pd.read_csv('/content/drive/MyDrive/GDELT/bq-results-20260526-220403-1779833103182.csv')

In [ ]:
df.head()

,GKGRECORDID,DATE,ArticleDate,Month,SourceCommonName,Host,RegisteredDomain,Headline,GDELT_AvgTone,GDELT_PositiveScore,...,V2Tone,V2Themes,V2Locations,V2Persons,V2Organizations,DocumentIdentifier,SourceGroup,UK_Relevance_Flag,Inflation_Relevance_Flag,Expectation_Language_Flag
0,20260327154500-1681,20260327154500,2026-03-27,2026-03,sky.com,news.sky.com,sky.com,All the things Donald Trump has put his name o...,3.302752,4.587156,...,"3.30275229357798,4.58715596330275,1.2844036697...","TAX_WEAPONS_WARSHIPS,2154;DEMOCRACY,1728;EPU_P...","3#Washington, Washington, United States#US#USD...","Donald Trump,52;Scott Bessent,315;John F Kenne...","Department Of Government Efficiency,1125;Donal...",https://news.sky.com/story/all-the-things-dona...,Media_Journalistic_Commentary,1,1,0
1,20250207124500-1063,20250207124500,2025-02-07,2025-02,sky.com,news.sky.com,sky.com,Billions for 'unproven' carbon capture technol...,0.228311,2.283105,...,"0.228310502283105,2.28310502283105,2.054794520...","MEDIA_SOCIAL,494;WB_135_TRANSPORT,1491;WB_1174...",1#United Kingdom#UK#UK##54#-4#UK#1537;1#United...,NaN,"Public Accounts Committee,350",https://news.sky.com/story/billions-for-unprov...,Media_Journalistic_Commentary,1,1,1
2,20260311150000-355,20260311150000,2026-03-11,2026-03,sky.com,news.sky.com,sky.com,Iran war: The outlook for your finances - whet...,-2.986162,1.019665,...,"-2.98616168973052,1.01966496722505,4.005826656...","MANMADE_DISASTER_IMPLIED,419;AFFECT,810;CRISIS...",1#United Kingdom#UK#UK##54#-4#UK#398;1#United ...,"Donald Trump,12;Lale Akoner,6528;Rachel Reeves...",NaN,https://news.sky.com/story/bluesky-13517570,Media_Journalistic_Commentary,1,1,1
3,20240924111500-1205,20240924111500,2024-09-24,2024-09,sky.com,news.sky.com,sky.com,Brent crude prices rise amid Israeli air strik...,-4.297994,0.286533,...,"-4.29799426934097,0.286532951289398,4.58452722...","TERROR,1796;ARMEDCONFLICT,1796;ECON_DIESELPRIC...",1#China#CH#CH##35#105#CH#403;1#United Kingdom#...,NaN,NaN,https://news.sky.com/story/brent-crude-prices-...,Media_Journalistic_Commentary,1,1,1
4,20250210200000-124,20250210200000,2025-02-10,2025-02,sky.com,news.sky.com,sky.com,Farmers' inheritance tax creates 'chilling eff...,-0.927152,3.178808,...,"-0.927152317880795,3.17880794701987,4.10596026...","WB_595_DRYLANDS,1703;WB_590_ECOSYSTEMS,1703;CR...","4#Whitehall, Orkney Islands, United Kingdom#UK...","Rachel Reeves,493;Richard Broadbent,214","Business Association,1070",https://news.sky.com/story/farmers-inheritance...,Media_Journalistic_Commentary,1,1,0


In [ ]:

for col in ["Headline", "V2Locations", "V2Themes", "V2Persons", "V2Organizations"]:
    df[col] = df[col].fillna("")

uk_text = (
    df["Headline"] + " " +
    df["V2Locations"] + " " +
    df["V2Persons"] + " " +
    df["V2Organizations"]
)

uk_pattern = (
    r"#UK#|"
    r"\bUnited Kingdom\b|"
    r"\bUK\b|"
    r"\bU\.K\.\b|"
    r"\bBritain\b|"
    r"\bBritish\b|"
    r"\bEngland\b|"
    r"\bScotland\b|"
    r"\bWales\b|"
    r"\bNorthern Ireland\b|"
    r"\bLondon\b|"
    r"\bBank of England\b|"
    r"\bONS\b|"
    r"\bOffice for National Statistics\b|"
    r"\bHM Treasury\b|"
    r"\bChancellor\b"
)

df["UK_Strict_Flag"] = uk_text.str.contains(
    uk_pattern,
    case=False,
    regex=True,
    na=False
)

In [ ]:
inflation_text = df["Headline"] + " " + df["V2Themes"]

core_inflation_pattern = (
    r"ECON_INFLATION|"
    r"WB_442_INFLATION|"
    r"ECON_COST_OF_LIVING|"
    r"TAX_ECON_PRICE|"
    r"FUELPRICES|"
    r"ECON_OILPRICE|"
    r"ENV_NATURALGAS|"
    r"ECON_INTEREST_RATES|"
    r"ECON_INTERESTRATES|"
    r"EPU_POLICY_INTEREST_RATE|"
    r"EPU_CATS_MONETARY_POLICY|"
    r"\binflation\b|"
    r"\bCPI\b|"
    r"\bcost of living\b|"
    r"\benergy bills?\b|"
    r"\bfood prices?\b|"
    r"\binterest rates?\b|"
    r"\bBank of England\b"
)

df["Core_Inflation_Flag"] = inflation_text.str.contains(
    core_inflation_pattern,
    case=False,
    regex=True,
    na=False
)

In [ ]:
df_scrape = df[
    (df["UK_Strict_Flag"] == True) &
    (df["Core_Inflation_Flag"] == True) &
    (df["Expectation_Language_Flag"] == 1)
].copy()

df_scrape = df_scrape.drop_duplicates(subset=["DocumentIdentifier"])

print(df_scrape.shape)
df_scrape.head()

(9410, 24)


,GKGRECORDID,DATE,ArticleDate,Month,SourceCommonName,Host,RegisteredDomain,Headline,GDELT_AvgTone,GDELT_PositiveScore,...,V2Locations,V2Persons,V2Organizations,DocumentIdentifier,SourceGroup,UK_Relevance_Flag,Inflation_Relevance_Flag,Expectation_Language_Flag,UK_Strict_Flag,Core_Inflation_Flag
1,20250207124500-1063,20250207124500,2025-02-07,2025-02,sky.com,news.sky.com,sky.com,Billions for 'unproven' carbon capture technol...,0.228311,2.283105,...,1#United Kingdom#UK#UK##54#-4#UK#1537;1#United...,,"Public Accounts Committee,350",https://news.sky.com/story/billions-for-unprov...,Media_Journalistic_Commentary,1,1,1,True,True
2,20260311150000-355,20260311150000,2026-03-11,2026-03,sky.com,news.sky.com,sky.com,Iran war: The outlook for your finances - whet...,-2.986162,1.019665,...,1#United Kingdom#UK#UK##54#-4#UK#398;1#United ...,"Donald Trump,12;Lale Akoner,6528;Rachel Reeves...",,https://news.sky.com/story/bluesky-13517570,Media_Journalistic_Commentary,1,1,1,True,True
3,20240924111500-1205,20240924111500,2024-09-24,2024-09,sky.com,news.sky.com,sky.com,Brent crude prices rise amid Israeli air strik...,-4.297994,0.286533,...,1#China#CH#CH##35#105#CH#403;1#United Kingdom#...,,,https://news.sky.com/story/brent-crude-prices-...,Media_Journalistic_Commentary,1,1,1,True,True
8,20241022143000-1169,20241022143000,2024-10-22,2024-10,sky.com,news.sky.com,sky.com,UK economic forecast boosted by IMF,-1.418440,2.304965,...,1#China#CH#CH##35#105#CH#3156;1#United Kingdom...,"Rachel Reeves,210","International Monetary Fund,31",https://news.sky.com/story/imf-upgrades-uk-by-...,Media_Journalistic_Commentary,1,1,1,True,True
19,20240618160000-616,20240618160000,2024-06-18,2024-06,sky.com,news.sky.com,sky.com,Money blog: Smoke machines deployed in Tesco; ...,1.111111,3.055556,...,"4#Sussex, East Sussex, United Kingdom#UK#UKE2#...","Teresa Payne,489;Teresa Payne,560;Parfitt Cres...","Cardiff University,1828",https://news.sky.com/story/money-blog-smoke-ma...,Media_Journalistic_Commentary,1,1,1,True,True


In [ ]:
!pip install newspaper3k pandas lxml_html_clean

In [ ]:
import pandas as pd
import re
from newspaper import Article
import time

In [ ]:
# text extraction symbol fix
def clean_text(text):
    if text:
        try:
            text = text.encode('latin1').decode('utf-8')
        except:
            pass
    return text

In [ ]:
df_scrape.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9410 entries, 1 to 89334
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   GKGRECORDID                9410 non-null   object 
 1   DATE                       9410 non-null   int64  
 2   ArticleDate                9410 non-null   object 
 3   Month                      9410 non-null   object 
 4   SourceCommonName           9410 non-null   object 
 5   Host                       9410 non-null   object 
 6   RegisteredDomain           9410 non-null   object 
 7   Headline                   9410 non-null   object 
 8   GDELT_AvgTone              9410 non-null   float64
 9   GDELT_PositiveScore        9410 non-null   float64
 10  GDELT_NegativeScore        9410 non-null   float64
 11  GDELT_Polarity             9410 non-null   float64
 12  V2Tone                     9410 non-null   object 
 13  V2Themes                   9410 non-null   object 
 

In [ ]:
# Create empty columns
df_scrape['Extracted_Text'] = None
df_scrape['Word_Count'] = 0

# Function to extract article text
def extract_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()

        text = clean_text(article.text)
        word_count = len(text.split())

        return text, word_count

    except:
        return None, 0

In [ ]:
# Loop through URLs
SAVE_INTERVAL = 1000 # Save every 1000 records
saved_count = 0

for i, (idx, url) in enumerate(df_scrape['DocumentIdentifier'].items()):
    text, wc = extract_text(url)

    df_scrape.loc[idx, 'Extracted_Text'] = text
    df_scrape.loc[idx, 'Word_Count'] = wc

    # Optionally print progress, be mindful of verbose output for large dataframes
    # print(f"Processed index {idx} / {len(df_scrape)} done")

    # Save incrementally
    if (i + 1) % SAVE_INTERVAL == 0:
        output_filename = f'/content/drive/MyDrive/GDELT/df_scrape_partial_save_{i + 1}.csv'
        df_scrape.to_csv(output_filename, index=False)
        print(f"Saved {i + 1} records to {output_filename}")
        saved_count = i + 1

    # Small delay to avoid blocking
    time.sleep(1)

# Save the final DataFrame after the loop, or the remaining records if the loop didn't hit a SAVE_INTERVAL evenly
if len(df_scrape) > saved_count:
    final_output_filename = f'/content/drive/MyDrive/GDELT/df_scrape_final_save.csv'
    df_scrape.to_csv(final_output_filename, index=False)
    print(f"All {len(df_scrape)} records processed. Final DataFrame saved to {final_output_filename}")

df_scrape.info()

Saved 1000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_1000.csv
Saved 2000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_2000.csv
Saved 3000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_3000.csv
Saved 4000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_4000.csv
Saved 5000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_5000.csv
Saved 6000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_6000.csv
Saved 7000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_7000.csv
Saved 8000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_8000.csv
Saved 9000 records to /content/drive/MyDrive/GDELT/df_scrape_partial_save_9000.csv
All 9410 records processed. Final DataFrame saved to /content/drive/MyDrive/GDELT/df_scrape_final_save.csv
<class 'pandas.core.frame.DataFrame'>
Index: 9410 entries, 1 to 89334
Data columns (total 26 columns):
 #   Column                     Non-Null Co

### Loading the final saved DataFrame

Given the discrepancy observed, it's best to load the `df_scrape_final_save.csv` directly, as it should contain the most complete set of extracted texts.

In [ ]:
# Load the final saved DataFrame directly
df_final_scrape = pd.read_csv('/content/drive/MyDrive/GDELT/df_scrape_final_save.csv')

print(f"Loaded final DataFrame shape: {df_final_scrape.shape}")
df_final_scrape.info()

Loaded final DataFrame shape: (9410, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9410 entries, 0 to 9409
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   GKGRECORDID                9410 non-null   object 
 1   DATE                       9410 non-null   int64  
 2   ArticleDate                9410 non-null   object 
 3   Month                      9410 non-null   object 
 4   SourceCommonName           9410 non-null   object 
 5   Host                       9410 non-null   object 
 6   RegisteredDomain           9410 non-null   object 
 7   Headline                   9410 non-null   object 
 8   GDELT_AvgTone              9410 non-null   float64
 9   GDELT_PositiveScore        9410 non-null   float64
 10  GDELT_NegativeScore        9410 non-null   float64
 11  GDELT_Polarity             9410 non-null   float64
 12  V2Tone                     9410 non-null   object 
 13  V2Theme